# 02 — Data Cleaning
## DataCo Supply Chain — Late Delivery Risk Prediction

**Goal of this notebook:**
- Apply all decisions made in 01_data_understanding.ipynb
- Drop useless, duplicate, leakage, and PII columns
- Fix data types
- Handle missing values
- Engineer new features: delay_gap, order_month, order_dayofweek, order_quarter
- Save the final cleaned dataset as cleaned_data.csv

**Input:**  data/DataCoSupplyChainDataset.csv  
**Output:** data/cleaned_data.csv  


In [10]:
# ===========================================
# Imports
# ===========================================
import pandas as pd
import numpy as np

## Load Raw Data

In [2]:
df = pd.read_csv('../data/DataCoSupplyChainDataset.csv',
                 encoding='latin-1')

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Shape: 180,519 rows × 53 columns


---
## Drop Unnecessary Columns
Based on decisions made in 01_data_understanding.ipynb
we drop columns in 4 categories: PII, Leakage, Duplicates, and Useless.

In [3]:
# ===========================================
# DROP COLUMNS
# ===========================================

# PII — personal information, no value for model
pii_cols = [
    'Customer Fname', 'Customer Lname', 'Customer Email',
    'Customer Password', 'Customer Street'
]

# LEAKAGE — only known after delivery
leakage_cols = [
    'Delivery Status'
    # Note: 'Days for shipping (real)' will be dropped AFTER engineering delay_gap
]

# DUPLICATES — identical or redundant columns
duplicate_cols = [
    'Product Category Id',      # same as Category Id
    'Order Item Cardprod Id',   # same as Product Card Id
    'Department Id',            # same as Department Name
    'Order Profit Per Order',   # same as Benefit per order
    'Sales per customer',       # same as Order Item Total
    'Order Item Total',         # derived
    'Order Item Discount',      # have Discount Rate instead
]

# USELESS — too granular, no value, or all null
useless_cols = [
    'Product Description',  # 100% null
    'Product Image',        # URL
    'Product Name',         # too granular, have Category Name
    'Product Status',       # only 1 unique value
    'Product Card Id',      # key only
    'Sales',                # derived
    'Order Customer Id',    # have Customer Segment
    'Order Item Id',        # table index
    'Customer City',        # store location not delivery
    'Customer State',       # store location not delivery
    'Customer Zipcode',     # too granular
    'Order City',           # too granular
    'Order State',          # too granular
    'Order Zipcode',        # too granular + nulls
    'Order Country',        # Market is better
]

# DROP ALL
cols_to_drop = pii_cols + leakage_cols + duplicate_cols + useless_cols
df.drop(columns=cols_to_drop, inplace=True)

print(f"Shape after dropping: {df.shape[0]:,} rows × {df.shape[1]} columns")

Shape after dropping: 180,519 rows × 25 columns


---
## Feature Engineering:delay_gap

Before dropping Days for shipping (real), We engineer delay_gap: This will be the target for Track 2 (Regression).

delay_gap = Days for shipping (real): Days for shipment (scheduled)

- Positive value = late delivery
- Zero = on time
- Negative value = early delivery

**Note:**
The column count stays at 25 because we drop one column
(Days for shipping real) and add one new column (delay_gap) at the same time.

In [4]:
# ===========================================
# ENGINEER delay_gap THEN DROP Days_real
# ===========================================

# Engineer delay_gap
df['delay_gap'] = (df['Days for shipping (real)'] - 
                   df['Days for shipment (scheduled)'])

print(f"delay_gap engineered")
print(f"Min  : {df['delay_gap'].min()}")
print(f"Max  : {df['delay_gap'].max()}")
print(f"Mean : {df['delay_gap'].mean():.2f}")

# Now drop Days for shipping (real) — leakage
df.drop(columns=['Days for shipping (real)'], inplace=True)

print(f"\nDays for shipping (real) dropped ")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

delay_gap engineered
Min  : -2
Max  : 4
Mean : 0.57

Days for shipping (real) dropped 
Shape: 180,519 rows × 25 columns


### delay_gap Results

- **Min = -2** → some orders arrived 2 days earlier than scheduled (early delivery)
- **Max = 4** → the worst case was 4 days later than scheduled
- **Mean = 0.57** → on average, orders are delivered about half a day late

 This means late deliveries are not extreme in terms of days,
 but they are frequent, which is exactly the **business problem** we are solving.

 Even a small delay consistently happening across thousands of orders
 creates a significant impact on **customer satisfaction and revenue.**
 

---

## Fix Data Types

In [ ]:
# ===========================================
# FIX DATA TYPES
# ===========================================
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'], errors='coerce')

df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'], errors='coerce')
 # errors='coerce' → if a value cannot be parsed, replace it with NaT instead of raising an error


print(f"order date type    : {df['order date (DateOrders)'].dtype}")
print(f"shipping date type : {df['shipping date (DateOrders)'].dtype}")

order date type    : datetime64[ns]
shipping date type : datetime64[ns]


---
## Extract Date Features
Extract useful features from date columns.
Instead of using the raw date, we extract:
- order_month → captures seasonal patterns
- order_dayofweek → captures weekly patterns (0=Monday, 6=Sunday)
- order_quarter → captures quarterly patterns

In [7]:
# ===========================================
# EXTRACT DATE FEATURES
# ===========================================
df['order_month']      = df['order date (DateOrders)'].dt.month
df['order_dayofweek']  = df['order date (DateOrders)'].dt.dayofweek
df['order_quarter']    = df['order date (DateOrders)'].dt.quarter

print(f"order_month     : {df['order_month'].unique()}")
print(f"order_dayofweek : {df['order_dayofweek'].unique()}")
print(f"order_quarter   : {df['order_quarter'].unique()}")


order_month     : [ 1  2 10  3 11 12  6  8  4  5  7  9]
order_dayofweek : [2 5 4 1 6 0 3]
order_quarter   : [1 4 2 3]


order_month → 12 full months 

order_dayofweek → 7 full days  (0=Monday, 6=Sunday)

order_quarter → 4 full quarters 

---

## Handle Missing Values
From 01_data_understanding.ipynb I found only 2 columns with nulls:
- Customer Lname → 8 nulls → already dropped (PII)
- Customer Zipcode → 3 nulls → already dropped (too granular)

Now I will check that **no missing** values remain.



In [8]:
missing = df.isnull().sum()
print(missing)

Type                             0
Days for shipment (scheduled)    0
Benefit per order                0
Late_delivery_risk               0
Category Id                      0
Category Name                    0
Customer Country                 0
Customer Id                      0
Customer Segment                 0
Department Name                  0
Latitude                         0
Longitude                        0
Market                           0
order date (DateOrders)          0
Order Id                         0
Order Item Discount Rate         0
Order Item Product Price         0
Order Item Profit Ratio          0
Order Item Quantity              0
Order Region                     0
Order Status                     0
Product Price                    0
shipping date (DateOrders)       0
Shipping Mode                    0
delay_gap                        0
order_month                      0
order_dayofweek                  0
order_quarter                    0
dtype: int64


### Result
All columns = 0 nulls 

No missing values ​​in the data

---
## Final Shape & Save


In [9]:
# ===========================================
# FINAL SHAPE
# ===========================================

print("CLEANING SUMMARY")
print("=" * 50)
print(f"Original shape : 180,519 rows × 53 columns")
print(f"Final shape    : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns removed: {53 - df.shape[1]}")
print(f"Missing values : {df.isnull().sum().sum()}")
print("=" * 50)

# Save
df.to_csv('../data/cleaned_data.csv', index=False)
print("cleaned_data.csv saved ")

CLEANING SUMMARY
Original shape : 180,519 rows × 53 columns
Final shape    : 180519 rows × 28 columns
Columns removed: 25
Missing values : 0
cleaned_data.csv saved 


---
## Summary

The raw dataset was reduced from 53 to 28 columns by removing:
- PII columns (names, email, password, street)
- Leakage columns (Delivery Status, Days for shipping real)
- Duplicate and redundant columns
- Useless columns (100% null, single value, too granular)

New engineered features added:
- delay_gap → target for Track 2 (Regression)
- order_month, order_dayofweek, order_quarter → time-based features

The cleaned dataset is saved as **data/cleaned_data.csv**
and is ready to be used by all 4 tracks.